In [ ]:
import pandas as pd
import numpy as np
import json
import joblib

from sklearn.model_selection import GridSearchCV, KFold
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

# ============================================================
# Load Data
# ============================================================

data = pd.read_csv(r'J:/CancerInst/ImmunoTherapy/Lab_Current/Guillaume Mestrallet/Experiments/MLmetastasis/msk_met_2021/dataML.csv')

X = data.drop(columns=["TIME_TO_MET_OR_NOT_MET_AFTER_SEQUENCING", "MET_STATUS"])
y = Surv.from_dataframe("MET_STATUS", "TIME_TO_MET_OR_NOT_MET_AFTER_SEQUENCING", data)

# ============================================================
# Hyperparameter Grid
# ============================================================

param_grid = {
    "n_estimators": [100, 500],
    "max_depth": [10, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [3, 5],
    "max_features": ["sqrt", None],
    "bootstrap": [True]
}

# ============================================================
# Step 1: Nested Cross-Validation
# ============================================================

outer_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

nested_c_indices = []
best_params_per_fold = []

for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X)):

    print(f"\n{'='*60}")
    print(f"OUTER FOLD {fold_idx + 1}")
    print(f"{'='*60}")

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    # ---------------- Inner CV ---------------- #

    inner_cv = KFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    rsf = RandomSurvivalForest(
        random_state=42,
        n_jobs=-1
    )

    grid_search = GridSearchCV(
        estimator=rsf,
        param_grid=param_grid,
        cv=inner_cv,
        scoring=None,   # RSF.score() = concordance index
        n_jobs=-1,
        refit=True
    )

    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_

    print("\nBest parameters:")
    print(grid_search.best_params_)

    best_params_per_fold.append(grid_search.best_params_)

    # ---------------- Evaluate on outer test fold ---------------- #

    y_pred_test = best_model.predict(X_test)

    c_index_test = concordance_index_censored(
        y_test["MET_STATUS"],
        y_test["TIME_TO_MET_OR_NOT_MET_AFTER_SEQUENCING"],
        y_pred_test
    )[0]

    print(f"Outer test C-index: {c_index_test:.4f}")

    nested_c_indices.append(c_index_test)

# ============================================================
# Nested CV Results
# ============================================================

mean_nested_cindex = np.mean(nested_c_indices)
std_nested_cindex = np.std(nested_c_indices)

print("\n")
print("=" * 60)
print("NESTED CROSS-VALIDATION RESULTS")
print("=" * 60)
print(f"Fold C-indices: {nested_c_indices}")
print(f"Mean Nested CV C-index: {mean_nested_cindex:.4f}")
print(f"Std Nested CV C-index : {std_nested_cindex:.4f}")
print("=" * 60)

# ============================================================
# Step 2: Final Hyperparameter Search on Full Dataset
# ============================================================

print("\nRunning final grid search on full dataset...")

final_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

final_grid_search = GridSearchCV(
    estimator=RandomSurvivalForest(
        random_state=42,
        n_jobs=-1
    ),
    param_grid=param_grid,
    cv=final_cv,
    scoring=None,
    n_jobs=-1,
    refit=True
)

final_grid_search.fit(X, y)

best_params = final_grid_search.best_params_

print("\nBest parameters from full dataset:")
print(best_params)

# ============================================================
# Step 3: Train Final Model
# ============================================================

final_model = RandomSurvivalForest(
    **best_params,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X, y)

print("\nFinal model trained on all samples.")

# Save the final model
joblib.dump(final_model, "final_rsf_model.pkl")

# ============================================================
# Save Results
# ============================================================

results = {
    "nested_cv_fold_c_indices": [float(x) for x in nested_c_indices],
    "nested_cv_mean_c_index": float(mean_nested_cindex),
    "nested_cv_std_c_index": float(std_nested_cindex),
    "best_params_per_outer_fold": best_params_per_fold,
    "final_best_params_full_dataset": best_params
}

with open("nested_cv_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("\nResults saved to nested_cv_results.json")

In [ ]:
best_model=final_model
best_model

In [ ]:
import shap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
explainer2 = shap.Explainer(best_model.predict, X)  # Approximate SHAP for predictions
shap_values2 = explainer2(X, max_evals=6500)
shap.summary_plot(shap_values2, X)

In [ ]:
X_test_sorted = X_test.sort_values(by=['CANCER_TYPE_Gastrointestinal Stromal Tumor','CANCER_TYPE_Breast Cancer','CANCER_TYPE_Prostate Cancer','cna_CDKN2B'])
X_test_sel = pd.concat((X_test_sorted.head(3), X_test_sorted.tail(3)))

X_test_sel

pd.Series(best_model.predict(X_test_sel))

surv = best_model.predict_survival_function(X_test_sel, return_array=True)

for i, s in enumerate(surv):
    plt.step(best_model.unique_times_, s, where="post", label=str(i))
plt.ylabel("Metastasis probability")
plt.xlabel("Time in years")
plt.legend()
plt.grid(True)

In [ ]:
X_test_sorted = X_test.sort_values(by=['CANCER_TYPE_Gastrointestinal Stromal Tumor','CANCER_TYPE_Breast Cancer','CANCER_TYPE_Prostate Cancer','cna_CDKN2B'])
X_test_sel = pd.concat((X_test_sorted.head(3), X_test_sorted.tail(3)))

X_test_sel

pd.Series(best_model.predict(X_test_sel))

surv = best_model.predict_cumulative_hazard_function(X_test_sel, return_array=True)

for i, s in enumerate(surv):
    plt.step(best_model.unique_times_, s, where="post", label=str(i))
plt.ylabel("Cumulative hazard")
plt.xlabel("Time in years")
plt.legend()
plt.grid(True)

In [ ]:
from sksurv.metrics import cumulative_dynamic_auc
import numpy as np

min_time = y_test["TIME_TO_MET_OR_NOT_MET_AFTER_SEQUENCING"].min()
max_time = y_test["TIME_TO_MET_OR_NOT_MET_AFTER_SEQUENCING"].max()

# Slightly shrink max time to stay strictly within interval
safe_max_time = max_time * 0.9999

# Generate safe time points
eval_times = np.linspace(min_time, safe_max_time, 10)

# Predict survival functions
pred_surv = best_model.predict_survival_function(X_test)

# Evaluate survival probabilities at the eval_times
pred_surv_matrix = np.asarray([[fn(t) for t in eval_times] for fn in pred_surv])
risk_scores = 1 - pred_surv_matrix

# Compute cumulative/dynamic AUC
cph_auc, mean_auc = cumulative_dynamic_auc(y_train, y_test, risk_scores, eval_times)

#plot
plt.plot(eval_times, cph_auc, marker='o', color='orange', label="data test")
plt.xlabel("Time (years)")
plt.ylabel("Time-dependent AUC")
plt.title("Time-dependent ROC AUC for RSF")
plt.ylim(0, 1)
plt.grid(True)
plt.show()

In [ ]:
from sksurv.nonparametric import kaplan_meier_estimator
import matplotlib.pyplot as plt

# Get event indicator and durations from test set
event = y_test["MET_STATUS"]
time = y_test["TIME_TO_MET_OR_NOT_MET_AFTER_SEQUENCING"]

# Compute KM estimate
km_time, km_survival = kaplan_meier_estimator(event, time)

# Predict survival functions for all test samples
pred_surv = best_model.predict_survival_function(X_test)

# Interpolate all survival functions to a common time grid
# Choose a common grid based on the KM time or a linspace
common_time_grid = np.linspace(0, time.max(), 100)

# Interpolate predicted survival values at these times
interp_surv = np.asarray([[fn(t) for t in common_time_grid] for fn in pred_surv])

# Compute mean predicted survival across all test samples
mean_pred_surv = interp_surv.mean(axis=0)

#plot
plt.step(km_time, km_survival, where="post", label="Kaplan–Meier (observed)")
plt.plot(common_time_grid, mean_pred_surv, label="RSF predicted (mean)", color="orange")
plt.xlabel("Time (years)")
plt.ylabel("Metastasis Probability")
plt.title("Observed vs Predicted Metastasis Curves")
plt.grid(True)
plt.legend()
plt.show()